# Reproducing Paper Figures

This notebook regenerates the 12 data-driven figures from the paper using the calibrated simulator. The 6 architectural figures (Figs 1–6) are TikZ-only and live in `paper/robin-fpga.tex`.

**Estimated runtime:** ~30 seconds.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = Path('../data/results')
assert DATA.exists(), 'run scripts/run_simulator.py first'

PAL = {
    'robin':  '#2E5C8A',
    'rust':   '#C2410C',
    'green':  '#15803D',
    'purple': '#6D28D9',
    'teal':   '#0F766E',
    'grey':   '#525252',
}
plt.rcParams.update({'font.family': 'sans-serif', 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})

## Figure 7: closure rate heatmap

In [ ]:
df = pd.read_csv(DATA / 'closure_rates.csv')
pivot = df.pivot(index='design', columns='baseline', values='closure_rate')
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0.4, vmax=1.0, aspect='auto')
ax.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=30, ha='right')
ax.set_yticks(range(len(pivot.index)), pivot.index)
fig.colorbar(im, ax=ax, label='closure rate')
ax.set_title('Closure rate across designs and baselines')
fig.tight_layout()
plt.show()

## Figure 8: convergence with confidence bands

In [ ]:
df = pd.read_csv(DATA / 'convergence.csv')
fig, ax = plt.subplots(figsize=(8, 5))
for prefix, color, label in [
    ('robin',  PAL['robin'], 'ROBIN-FPGA'),
    ('drills', PAL['rust'],  'DRiLLS-style DQN'),
    ('turbo',  PAL['grey'],  'TuRBO'),
]:
    ax.plot(df['episode'], df[f'{prefix}_mean'], color=color, label=label, lw=1.6)
    ax.fill_between(df['episode'], df[f'{prefix}_p10'], df[f'{prefix}_p90'], color=color, alpha=0.18)
ax.axhline(0, color=PAL['grey'], lw=0.6, ls='--', alpha=0.6)
ax.set_xlabel('training episode')
ax.set_ylabel(r'CVaR$_{0.2}$ slack-normalised return')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.25)
ax.set_title('Convergence on GEMM-systolic-16x16')
plt.show()

## Figure 11: conformal coverage

In [ ]:
df = pd.read_csv(DATA / 'coverage.csv')
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(df['alpha'], df['target'], color=PAL['grey'], ls='--', label=r'target $1-\alpha$')
ax.plot(df['alpha'], df['exch_coverage'], 'o-', color=PAL['green'], label='exchangeable calibration')
ax.plot(df['alpha'], df['drift_coverage'], 's-', color=PAL['rust'], label='tool minor-version drift')
ax.set_xlabel(r'$\alpha$')
ax.set_ylabel('empirical coverage')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.25)
ax.set_title('Conformal envelope coverage')
plt.show()

## Figure 12: Pareto frontier

In [ ]:
df = pd.read_csv(DATA / 'pareto.csv')
fig, ax = plt.subplots(figsize=(7, 5))
for method, color, marker in [
    ('ROBIN-FPGA', PAL['robin'],  'o'),
    ('DRiLLS',     PAL['rust'],   's'),
    ('TuRBO',      PAL['grey'],   '^'),
    ('Defaults',   PAL['purple'], 'x'),
]:
    sub = df[df['method'] == method]
    ax.scatter(sub['latency_ns'], sub['power_W'], color=color, marker=marker, s=40, alpha=0.75, label=method)
ax.set_xlabel('critical-path latency [ns]')
ax.set_ylabel('dynamic power [W]')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.25)
ax.set_title('Pareto frontier on GEMM-systolic')
plt.show()

## All figures via the script

To regenerate the full set of PNG previews, run the convenience script:

```bash
python ../scripts/generate_figures.py --data ../data/results/ --output ../paper/figures/
```